<a href="https://colab.research.google.com/github/vyshnavi7-coder/C-PROGRAMMING/blob/main/amazon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Create all required folders
folders = [
    "Student_resource/dataset/train",
    "Student_resource/dataset/test",
    "Student_resource/output",
    "Student_resource/utils"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [ ]:
import os
import shutil

train_folder = "Student_resource/dataset/train"
os.makedirs(train_folder, exist_ok=True)

train_files = [
    "train_ground_truth.tsv",
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
]

for f in train_files:
    source_path = os.path.join("Student_resource/dataset", f)
    dest_path = os.path.join(train_folder, f)
    if os.path.exists(source_path):
        shutil.move(source_path, dest_path)

print("Files moved into the train folder successfully!")

Files moved into the train folder successfully!


In [ ]:
import pandas as pd

df = pd.read_csv("Student_resource/dataset/train/train_source1.tsv", sep="\t")
df.head()

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


In [ ]:
import os
import re
import pandas as pd

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

# Create output folder
output_dir = "Student_resource/output"
os.makedirs(output_dir, exist_ok=True)

print("Loading datasets...")
s1 = pd.read_csv("Student_resource/dataset/train/train_source1.tsv", sep="\t")
s2 = pd.read_csv("Student_resource/dataset/train/train_source2.tsv", sep="\t")
s3 = pd.read_csv("Student_resource/dataset/train/train_source3.tsv", sep="\t")

# Combine Source 2 and Source 3
s2["source"] = "S2"
s3["source"] = "S3"
s23 = pd.concat([s2, s3], ignore_index=True)
print("Data loaded successfully!")

Loading datasets...
Data loaded successfully!


Preprocess Text Fields




In [ ]:
print("Preprocessing business names...")
for df in [s1, s23]:
    df["clean_name"] = df["business_name"].apply(clean_text)
    df["name_token"] = df["clean_name"].apply(
        lambda x: x.split()[0] if len(x.split()) > 0 else ""
    )

print("Text preprocessing complete!")

Preprocessing business names...
Text preprocessing complete!


Build Blocking Indexes

In [ ]:
print("Building block index...")
block_index = {}
for _, row in s23.iterrows():
    country = str(row["country"]).strip().lower()
    token = row["name_token"]
    if not token:
        continue
    key = (country, token)
    if key not in block_index:
        block_index[key] = []
    block_index[key].append(row["entity_id"])

print("Building country fallback index...")
country_index = {}
for _, row in s23.iterrows():
    country = str(row["country"]).strip().lower()
    if country not in country_index:
        country_index[country] = []
    country_index[country].append(row["entity_id"])

print("Indexes built successfully!")

Generate Candidate Pairs

In [ ]:
print("Generating candidate pairs (this may take a few moments)...")
candidate_rows = []

for _, row in s1.iterrows():
    s1_id = row["entity_id"]
    country = str(row["country"]).strip().lower()
    clean_name = row["clean_name"]
    tokens = clean_name.split()

    candidates = set()

    # Strategy A: Match by Country + First/Second Name Token
    if tokens:
        primary_token = tokens[0]
        key = (country, primary_token)
        if key in block_index:
            candidates.update(block_index[key])

        if len(tokens) > 1:
            secondary_token = tokens[1]
            key_sec = (country, secondary_token)
            if key_sec in block_index:
                candidates.update(block_index[key_sec])

    # Strategy B: Fallback to country-level block if empty
    if len(candidates) == 0 and country in country_index:
        candidates.update(country_index[country])

    candidate_str = ",".join(sorted(list(candidates)))
    candidate_rows.append(
        {"source1_entity_id": s1_id, "candidate_entity_ids": candidate_str}
    )

print("Candidates generated!")

Save Output File

In [ ]:
output_df = pd.DataFrame(candidate_rows)
output_path = os.path.join(output_dir, "candidate_pairs.tsv")
output_df.to_csv(output_path, sep="\t", index=False)

print(f"Successfully generated {output_path} with {len(output_df)} rows.")
output_df.head()

II.Imports and Helper Functions

In [ ]:
import os
import re
import pandas as pd
from rapidfuzz import fuzz
from sklearn.ensemble import RandomForestClassifier


def clean_text(text):
  if pd.isna(text):
    return ""
  text = str(text).lower()
  text = re.sub(r"[^\w\s]", " ", text)
  return re.sub(r"\s+", " ", text).strip()


def compute_features(row, s1_dict, s23_dict):
  s1_rec = s1_dict.get(row["source1_entity_id"], {})
  s23_rec = s23_dict.get(row["candidate_entity_id"], {})

  name1 = s1_rec.get("clean_name", "")
  name2 = s23_rec.get("clean_name", "")

  addr1 = s1_rec.get("clean_address", "")
  addr2 = s23_rec.get("clean_address", "")

  country_match = 1 if s1_rec.get("country", "") == s23_rec.get("country", "") else 0

  # Compute string similarity metrics
  name_ratio = fuzz.ratio(name1, name2) / 100.0
  name_partial = fuzz.partial_ratio(name1, name2) / 100.0
  name_token_sort = fuzz.token_sort_ratio(name1, name2) / 100.0

  addr_ratio = fuzz.ratio(addr1, addr2) / 100.0
  addr_token_sort = fuzz.token_sort_ratio(addr1, addr2) / 100.0

  return {
      "country_match": country_match,
      "name_ratio": name_ratio,
      "name_partial": name_partial,
      "name_token_sort": name_token_sort,
      "addr_ratio": addr_ratio,
      "addr_token_sort": addr_token_sort,
  }

Loading & Preprocessing Training Data

In [ ]:
os.makedirs("Student_resource/output", exist_ok=True)

print("Loading datasets...")
# Load train source datasets and ground truth
s1_train = pd.read_csv("Student_resource/dataset/train/train_source1.tsv", sep="\t")
s2_train = pd.read_csv("Student_resource/dataset/train/train_source2.tsv", sep="\t")
s3_train = pd.read_csv("Student_resource/dataset/train/train_source3.tsv", sep="\t")
gt = pd.read_csv("Student_resource/dataset/train/train_ground_truth.tsv", sep="\t")

s2_train["source"] = "S2"
s3_train["source"] = "S3"
s23_train = pd.concat([s2_train, s3_train], ignore_index=True)

# Preprocess text
for df in [s1_train, s23_train]:
  df["clean_name"] = df["business_name"].apply(clean_text)
  df["clean_address"] = df["business_address"].apply(clean_text)
  df["country"] = df["country"].astype(str).str.strip().str.lower()

s1_train_dict = s1_train.set_index("entity_id").to_dict("index")
s23_train_dict = s23_train.set_index("entity_id").to_dict("index")

Building Training Pairs & Training the Classifier

In [ ]:
print("Building training pairs from candidates and ground truth...")
# Generate candidate pairs from the training split or load candidate_pairs.tsv if already generated
# For training, we pair each S1 entity with its blocking candidates or construct pairs via ground truth positives + hard negatives
gt_mapping = {}
for _, row in gt.iterrows():
  matches = str(row["matched_entity_ids"]).strip()
  if matches and matches != "nan":
    gt_mapping[row["source1_entity_id"]] = set(matches.split(","))
  else:
    gt_mapping[row["source1_entity_id"]] = set()

training_rows = []
for s1_id, true_matches in gt_mapping.items():
  s1_country = s1_train_dict.get(s1_id, {}).get("country", "")
  # Add true matches
  for m_id in true_matches:
    training_rows.append(
        {
            "source1_entity_id": s1_id,
            "candidate_entity_id": m_id,
            "label": 1,
        }
    )
  # Add a sample of non-matches from the same country as negative examples
  same_country_s23 = [
      eid
      for eid, rec in s23_train_dict.items()
      if rec["country"] == s1_country and eid not in true_matches
  ]
  # Sample up to 3 negatives per positive/entity to keep dataset balanced
  import random

  sampled_negatives = random.sample(
      same_country_s23, min(len(same_country_s23), max(1, len(true_matches) * 3))
  )
  for m_id in sampled_negatives:
    training_rows.append(
        {
            "source1_entity_id": s1_id,
            "candidate_entity_id": m_id,
            "label": 0,
        }
    )

train_pairs_df = pd.DataFrame(training_rows)

print("Computing features for training data...")
train_features = []
for _, row in train_pairs_df.iterrows():
  feats = compute_features(row, s1_train_dict, s23_train_dict)
  feats["label"] = row["label"]
  train_features.append(feats)

feat_df = pd.DataFrame(train_features)
X = feat_df.drop(columns=["label"])
y = feat_df["label"]

print("Training Random Forest Classifier...")
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
model.fit(X, y)

Loading & Preprocessing Test Data

In [ ]:
print("Loading test datasets and candidates...")
# Note: Ensure candidate_pairs.tsv was generated for test set prior to running inference
if not os.path.exists("Student_resource/output/candidate_pairs.tsv"):
  raise FileNotFoundError(
      "Student_resource/output/candidate_pairs.tsv not found. Run your blocking script first!"
  )

candidates_df = pd.read_csv("Student_resource/output/candidate_pairs.tsv", sep="\t")

s1_test = pd.read_csv("Student_resource/dataset/test/test_source1.tsv", sep="\t")
s2_test = pd.read_csv("Student_resource/dataset/test/test_source2.tsv", sep="\t")
s3_test = pd.read_csv("Student_resource/dataset/test/test_source3.tsv", sep="\t")

s2_test["source"] = "S2"
s3_test["source"] = "S3"
s23_test = pd.concat([s2_test, s3_test], ignore_index=True)

for df in [s1_test, s23_test]:
  df["clean_name"] = df["business_name"].apply(clean_text)
  df["clean_address"] = df["business_address"].apply(clean_text)
  df["country"] = df["country"].astype(str).str.strip().str.lower()

s1_test_dict = s1_test.set_index("entity_id").to_dict("index")
s23_test_dict = s23_test.set_index("entity_id").to_dict("index")

Inference, Thresholding, and Saving Output

In [ ]:
print("Running inference and applying precision-tuned threshold...")
matching_results = []
# Threshold set high (e.g., 0.8) to favor precision for F_0.5 optimization
THRESHOLD = 0.80

for _, row in candidates_df.iterrows():
  s1_id = row["source1_entity_id"]
  cand_str = str(row["candidate_entity_ids"])
  if not cand_str or cand_str == "nan":
    matching_results.append(
        {"source1_entity_id": s1_id, "matched_entity_ids": ""}
    )
    continue

  cand_ids = cand_str.split(",")
  pair_rows = [
      {"source1_entity_id": s1_id, "candidate_entity_id": cid} for cid in cand_ids
  ]

  test_feats = [compute_features(pr, s1_test_dict, s23_test_dict) for pr in pair_rows]
  test_feat_df = pd.DataFrame(test_feats)

  if not test_feat_df.empty:
    probs = model.predict_proba(test_feat_df)[:, 1]
    matched_cands = [
        cid for cid, prob in zip(cand_ids, probs) if prob >= THRESHOLD
    ]
    matching_results.append(
        {
            "source1_entity_id": s1_id,
            "matched_entity_ids": ",".join(matched_cands),
        }
    )
  else:
    matching_results.append(
        {"source1_entity_id": s1_id, "matched_entity_ids": ""}
    )

output_df = pd.DataFrame(matching_results)
output_path = "Student_resource/output/matching_results.tsv"
output_df.to_csv(output_path, sep="\t", index=False)
print(f"Successfully generated {output_path} with {len(output_df)} rows.")

Use ! at the start of a command to run terminal bash scripts in Colab

In [ ]:
!python3 Student_resource/utils/validate_submission.py --matching Student_resource/output/matching_results.tsv --candidate Student_resource/output/candidate_pairs.tsv --test-dir Student_resource/dataset/test